# Quantum Fourier transform

The quantum Fourier transform, QFT, is one of the most used constructions in quantum algorithms: it appears inside phase estimation, Shor's factoring algorithm, and other algorithms outside the scope of this series. This notebook builds it for three qubits and checks that it works by undoing it with its inverse.

## What it does, in words

The QFT transforms a computational-basis state into a superposition where the original information is encoded in the relative phases between qubits, instead of in the directly measured values. That is why this notebook does not verify the result by looking at the counts of a single QFT: phases do not show up that way. Instead, the QFT is applied followed by its inverse: if the circuit is correct, the state returns exactly to where it started.

## Building the QFT

It is built with the H and CP gates already seen in [`02a`](02a_gates_and_circuits.ipynb), plus a final qubit swap with SWAP to leave them in the right order:

In [ ]:
import math

import polypus

qc = polypus.Circuit(3)
qc.x(0)  # prepare a concrete starting state, |101>
qc.x(2)

# forward QFT
qc.h(0)
qc.cp(1, 0, math.pi / 2)
qc.cp(2, 0, math.pi / 4)
qc.h(1)
qc.cp(2, 1, math.pi / 2)
qc.h(2)
qc.swap(0, 2)

Each CP gate adds a phase that depends on the distance between the two qubits: the farther apart, the smaller the angle. The final SWAP undoes the reversed order the rest of the circuit leaves behind.

With the same pattern from [`02b`](02b_qiskit_qasm_interop.ipynb), the circuit up to this point can be seen:

In [ ]:
from qiskit import qasm2

qc_drawing = qasm2.loads(
    qc.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_drawing.draw("mpl")

## Undoing it with the inverse

The QFT is a unitary gate, so its inverse is the same gates, in reverse order, with angles of the opposite sign:

In [ ]:
qc.swap(0, 2)
qc.h(2)
qc.cp(2, 1, -math.pi / 2)
qc.h(1)
qc.cp(2, 0, -math.pi / 4)
qc.cp(1, 0, -math.pi / 2)
qc.h(0)

qc.measure_all()

## Checking the result

In [ ]:
result = polypus.run_quantum_circuit(qc, shots=2000, infrastructure="local")
print(result.counts[0])

The result is `'101'` almost 100% of the time, the same starting state: the QFT and its inverse cancel out exactly, just like the OpenQASM round trip checked in [`02b`](02b_qiskit_qasm_interop.ipynb).

## Summary

This notebook built the quantum Fourier transform for three qubits with H, CP, and SWAP, and checked it is correct by undoing it with its inverse instead of directly interpreting its counts, because the information it encodes is in phases, not in directly measurable values.

The next section covers the first algorithms with a provable advantage: Deutsch-Jozsa and Grover.

## Next step

Continue with: [`05a_deutsch_jozsa.ipynb`](05a_deutsch_jozsa.ipynb).